This is firs assignment (out of three). 

Submission guideline:

- Total point- 100 

- Attend all questions.

- Upload a zip/rar file containing the following:

    - MAS512_assignment1_gr#.py or MAS512_assignment1_gr#.ipynb. These files should be error free.

    - Each questions should be done in a cell, such that when I run the cell - I get the outputs.
    
    - MAS512_assignment1_gr#.ppt explaining the objective, method and results of each questions. 
    
    - Make   the presentation clear, readable, precise and attractive.

- Each group will be given 10 min to present the assignment followed by 5 minutes of Q/A. The presentation takes place in Lab, and all students are expected to attend peer's presentation

- Assignment is graded based on problem solving skills, work break down, algorithms, code, powerpoint and presentation

- Students are expected to solve the problem themselves.

- You can use LLM generative models as 'help', understand the working and algorithm, modify and use them.

- Be specific and precise when you write code, make powerpoint and present assignment


# Q1. Object detection and depth estimation                             20+5+20 = 45 points
 Use L515 camera in Lab to acquire rgb video and depth video (e.g. 30 s) of swinging load in lab (rgb_train_val_swingingload.avi; depth_train_val_swingingload.avi). Extract each frames from rgb video. Use the extracted images to train and validate YOLO model (save as YOLO_swinging_load.pt). Record test_swingingload.avi (e.g. 5 s). Extract frames and use them as test data set. Test your YOLO model (YOLO_swinging_load.pt) on this test data set.

(a) Show the performance of your model on test data set by plotting relevant KPIs.

(b) Sample five random images from test data video (test_swingingload.avi). Find the center, and coordinates of bounding box. Display them on the each test image.

(c) realsense_acq.ipynb contains code that streams, rgb, and depth video. Based on this code, extract the frames from depth_train_val_swingingload.avi. Each pixel in each frame consists of depth information of corresponding rgb images. As described in Q1.b find the center of bounding box of ALL frames in  test video (test_swingingload.avi). Display its center and corresponding depth.


References: Use Ultralytics framework for help - https://docs.ultralytics.com/tasks/detect
Realsense - https://dev.realsenseai.com/sdk-2-0-code-samples-wrappers-and-languages/opencv/ 

In [ ]:
# Task A: Train YOLOv11n on the provided dataset and evaluate its performance
import csv
from pathlib import Path
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
from ultralytics import YOLO

HERE = Path.cwd()
DATA = HERE / "assignment1.v2i.yolov11" / "data.yaml"  # dataset yaml file
model = YOLO("yolo11n.pt")  # pretrained COCO weights

results = model.train(
    data=str(DATA),
    epochs=100,          # how many epochs to train for
    imgsz=640,          # dataset is 640x640
    batch=16,           # images per batch
    device=0,           # use "cpu" if CUDA is unavailable
    workers=4,
    patience=25,        # early stop if val mAP stalls (only matters if epochs > 25)
    project=str(HERE / "runs"),
    name="q1_model",  # saved in runs/yolo11n_load
    exist_ok=False,      # not overwrite the same folder on rerun
    seed=0,
)

save_dir = Path(results.save_dir)
results_csv = save_dir / "results.csv"  # per-epoch metrics written during training

with open(results_csv, newline="") as f:
    reader = csv.DictReader(f)
    reader.fieldnames = [name.strip() for name in reader.fieldnames]  # some versions pad headers
    rows = [{k: v.strip() for k, v in row.items()} for row in reader]

def col(name):
    return [float(r[name]) for r in rows]

epochs = col("epoch")

# Training vs validation loss
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
loss_pairs = [
    ("train/box_loss", "val/box_loss", "Box loss"),
    ("train/cls_loss", "val/cls_loss", "Class loss"),
    ("train/dfl_loss", "val/dfl_loss", "DFL loss"),
]
for ax, (train_col, val_col, title) in zip(axes, loss_pairs):
    ax.plot(epochs, col(train_col), label="train")
    ax.plot(epochs, col(val_col), label="val")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(alpha=0.3)
fig.suptitle("Training vs Validation Loss")
fig.tight_layout()
plt.show()

# mAP50 and mAP50-95
plt.figure(figsize=(8, 5))
plt.plot(epochs, col("metrics/mAP50(B)"), label="mAP50")
plt.plot(epochs, col("metrics/mAP50-95(B)"), label="mAP50-95")
plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.title("Validation mAP over Training")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Precision and recall
plt.figure(figsize=(8, 5))
plt.plot(epochs, col("metrics/precision(B)"), label="Precision")
plt.plot(epochs, col("metrics/recall(B)"), label="Recall")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Precision and Recall over Training")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Learning rate schedule
plt.figure(figsize=(8, 5))
for lr_col in ["lr/pg0", "lr/pg1", "lr/pg2"]:
    if lr_col in rows[0]:
        plt.plot(epochs, col(lr_col), label=lr_col)
plt.xlabel("Epoch")
plt.ylabel("Learning rate")
plt.title("Learning Rate Schedule")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Final test metrics (model now holds best.pt)
metrics = model.val(data=str(DATA), split="test")
print(f"test mAP50    : {metrics.box.map50:.4f}")
print(f"test mAP50-95 : {metrics.box.map:.4f}")
print(f"test precision: {metrics.box.mp:.4f}")
print(f"test recall   : {metrics.box.mr:.4f}")

# Test metrics bar chart
labels = ["Precision", "Recall", "mAP50", "mAP50-95"]
values = [metrics.box.mp, metrics.box.mr, metrics.box.map50, metrics.box.map]
plt.figure(figsize=(6, 4))
bars = plt.bar(labels, values)
plt.bar_label(bars, fmt="%.3f")
plt.ylim(0, 1)
plt.title("Test Metrics")
plt.grid(axis="y", alpha=0.3)
plt.show()

# PR curve and Ultralytics' summary plot
for img_name in ["PR_curve.png", "results.png"]:
    img_path = save_dir / img_name
    if img_path.exists():
        plt.figure(figsize=(10, 8))
        plt.imshow(mpimg.imread(img_path))
        plt.axis("off")
        plt.title(img_name)
        plt.show()

New https://pypi.org/project/ultralytics/8.4.165 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.253 🚀 Python-3.10.20 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 5806MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/havard/hand_in/MAS512_Assignment_1/assignment1.v2i.yolov11/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosai

<Figure size 1500x400 with 3 Axes>

<Figure size 800x500 with 1 Axes>

<Figure size 800x500 with 1 Axes>

<Figure size 800x500 with 1 Axes>

Ultralytics 8.3.253 🚀 Python-3.10.20 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 5806MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 150.8±52.9 MB/s, size: 54.3 KB)
val: Scanning /home/havard/hand_in/MAS512_Assignment_1/assignment1.v2i.yolov11/test/labels... 150 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 150/150 3.3Kit/s 0.0s
val: New cache created: /home/havard/hand_in/MAS512_Assignment_1/assignment1.v2i.yolov11/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 7.5it/s 1.3s.2s
                   all        150        150          1          1      0.995      0.854
Speed: 1.4ms preprocess, 4.0ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /home/havard/hand_in/MAS512_Assignment_1/runs/detect/val
test mAP50    : 0.9950
test mAP50-95 : 0.8541
test precision: 0.9996
test

<Figure size 600x400 with 1 Axes>

<Figure size 1000x800 with 1 Axes>

In [5]:
# TASK B: sample five random frames from the test video, run the trained YOLo model on them, and display the bounding box + its centre on each image.

import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

HERE = Path.cwd()

# Test video recorded in the lab - its frames are the test data set.
# Found by search so the cell keeps working if the data folder is reorganised.
VIDEO_PATH = HERE / "bin" / "ajit_test_data" / "color_video" / "color_video.avi"
N_SAMPLES = 5
CONF = 0.50    # detection threshold
SEED = 0           # reproducible sample

# Weights: the Task A run we present (run 2, 100 epochs).
WEIGHTS = HERE / "runs" / "q1_model" / "weights" / "best.pt"

print(f"Weights: {Path(WEIGHTS).relative_to(HERE)}")
inference_model = YOLO(str(WEIGHTS))

# Select 5 semples
cap = cv2.VideoCapture(str(VIDEO_PATH))

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
random.seed(SEED)
sample_indices = sorted(random.sample(range(total_frames), min(N_SAMPLES, total_frames)))

# Keep the frames in BGR, ultralytics expects BGR for numpy input 
frames = {}
for idx in sample_indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame_bgr = cap.read()
    if ok:
        frames[idx] = frame_bgr
cap.release()

# Detect and give back boxes, confidence, label and centre of the bounding box for each frame
for idx, frame_bgr in frames.items():
    result = inference_model(frame_bgr, conf=CONF, verbose=False)[0]

    fig, ax = plt.subplots(figsize=(8, 6)) #Displaying output
    ax.imshow(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))  # matplotlib expects RGB so convergt BGR to RGB for display
    ax.set_title(f"Test frame {idx}")
    ax.axis("off")

    for box in result.boxes: #detect and annotate each bounding box in the frame
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        conf = float(box.conf[0])
        label = inference_model.names[int(box.cls[0])]
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2      # centre of the bounding box

        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="green", linewidth=2))#make rectangle around the detected object
        ax.plot(cx, cy, "ro", markersize=6)
        ax.text(x1, max(y1 - 8, 0),
                f"{label} {conf:.2f}\n"
                f"box=({x1:.0f},{y1:.0f})-({x2:.0f},{y2:.0f})\n"
                f"center=({cx:.0f},{cy:.0f})",
                color="white", fontsize=9,
                bbox=dict(facecolor="black", alpha=0.6, pad=2))
        #print the detection information to the console
        print(f"Frame {idx}: {label} conf={conf:.2f} "
              f"box=(x1={x1:.1f}, y1={y1:.1f}, x2={x2:.1f}, y2={y2:.1f}) "
              f"center=({cx:.1f}, {cy:.1f})")

    plt.tight_layout()
    plt.show() # show the plots

Weights: runs/q1_model/weights/best.pt
Frame 20: Load conf=0.90 box=(x1=364.9, y1=229.2, x2=442.3, y2=277.6) center=(403.6, 253.4)


<Figure size 800x600 with 1 Axes>

Frame 132: Load conf=0.90 box=(x1=354.8, y1=137.4, x2=466.6, y2=241.9) center=(410.7, 189.6)


<Figure size 800x600 with 1 Axes>

Frame 197: Load conf=0.90 box=(x1=358.8, y1=226.9, x2=431.3, y2=275.3) center=(395.1, 251.1)


<Figure size 800x600 with 1 Axes>

Frame 215: Load conf=0.92 box=(x1=349.0, y1=118.1, x2=450.6, y2=231.1) center=(399.8, 174.6)


<Figure size 800x600 with 1 Axes>

Frame 261: Load conf=0.65 box=(x1=364.5, y1=158.6, x2=464.2, y2=252.3) center=(414.4, 205.5)


<Figure size 800x600 with 1 Axes>

In [11]:
#Task C: realsense_acq.ipynb contains code that streams, rgb, and depth video. Based on this code, extract the frames from depth_train_val_swingingload.avi.
#  Each pixel in each frame consists of depth information of corresponding rgb images. As described in Q1.b find the center of bounding box of ALL frames in 
#  test video (test_swingingload.avi). Display its center and corresponding depth.

import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

HERE = Path.cwd()
SESSION = HERE / "bin" / "ajit_test_data"
VIDEO = next((SESSION / "color_video").glob("*.avi"))            # video to be used
DEPTH_DIR = SESSION / "depth_raw"                                # z16 frames, aligned to colour for depth
DEPTH_SCALE = json.loads((SESSION / "depth_scale_json" / "depth_scale.json").read_text())["depth_scale"]

model = YOLO(str(HERE / "runs" / "q1_model" / "weights" / "best.pt")) #run 2 was good
PATCH = 7          # half-size of the depth patch read at the centre in pixels
PREVIEW_EVERY = 60  # show an rgb + depth pair every 2 s (30 fps)

centres, depths, previews = [], [], []

# Ultralytics reads the video itself, so the frames keep the BGR order the model
# was trained on (same as Q2).
for i, result in enumerate(model.predict(source=str(VIDEO), stream=True, conf=0.25, verbose=False)):
    depth = np.load(DEPTH_DIR / f"depth_{i:04d}_raw.npy") * DEPTH_SCALE   # metres
    box = result.boxes

    if len(box) == 0:#if no bounding box is detected, append NaN values to the centres and depths lists and print a message indicating no detection for the current frame
        centres.append((np.nan, np.nan))
        depths.append(np.nan)
        print(f"frame {i:3d}   no detection")
        continue

    x1, y1, x2, y2 = box.xyxy[int(box.conf.argmax())].tolist()   # one load in the scene
    u = int((x1 + x2) / 2) # centre of the bounding box
    v = int((y1 + y2) / 2) # centre of the bounding box

    patch = depth[max(v - PATCH, 0):v + PATCH + 1, max(u - PATCH, 0):u + PATCH + 1]
    valid = patch[patch > 0] # 0 = sensor got no return
    Z = float(np.median(valid)) if valid.size else np.nan

    centres.append((u, v))
    depths.append(Z)
    print(f"frame {i:3d}   centre=({u:3d}, {v:3d})   depth={Z:.3f} m")

    if i % PREVIEW_EVERY == 0:
        previews.append((i, result.orig_img.copy(), depth, u, v, Z))

centres = np.array(centres, dtype=float)
depths = np.array(depths, dtype=float)
print(f"\n{len(depths)} frames | load detected in {np.isfinite(centres[:, 0]).sum()} | "
      f"depth {np.nanmin(depths):.3f} - {np.nanmax(depths):.3f} m")

# centre and depth over the whole test video 
fig, (ax_uv, ax_z) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
ax_uv.plot(centres[:, 0], label="u (horizontal)")
ax_uv.plot(centres[:, 1], label="v (vertical)")
ax_uv.set(ylabel="Centre [px]", title="Q1.c - bounding box centre and its depth, all test frames")
ax_uv.legend()
ax_z.plot(depths, color="tab:red")
ax_z.set(xlabel="Frame", ylabel="Depth at centre [m]")
for ax in (ax_uv, ax_z):
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

# rgb and depth pairs (colormap as in realsense_acq.ipynb)
for i, frame, depth, u, v, Z in previews:
    depth = np.where(depth > 0, depth, np.nan)       # blank the pixels with no reading
    fig, (ax_rgb, ax_depth) = plt.subplots(1, 2, figsize=(13, 5))

    ax_rgb.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax_rgb.plot(u, v, "ro")
    ax_rgb.set_title(f"Frame {i} - centre ({u}, {v}), depth {Z:.3f} m")

    im = ax_depth.imshow(depth, cmap="jet", vmin=np.nanpercentile(depth, 1), vmax=np.nanpercentile(depth, 99))
    ax_depth.plot(u, v, "wo", markeredgecolor="black")
    ax_depth.set_title("Aligned depth")
    fig.colorbar(im, ax=ax_depth, fraction=0.046, pad=0.04).set_label("Depth (m)")

    for ax in (ax_rgb, ax_depth):
        ax.axis("off")
    fig.tight_layout()
    plt.show() #show the images

frame   0   centre=(399, 189)   depth=2.810 m
frame   1   centre=(423, 242)   depth=3.403 m
frame   2   centre=(422, 241)   depth=3.511 m
frame   3   centre=(424, 244)   depth=3.612 m
frame   4   centre=(423, 248)   depth=3.726 m
frame   5   centre=(423, 251)   depth=3.812 m
frame   6   centre=(423, 252)   depth=3.867 m
frame   7   centre=(421, 253)   depth=3.927 m
frame   8   centre=(421, 253)   depth=3.964 m
frame   9   centre=(421, 255)   depth=3.948 m
frame  10   centre=(421, 255)   depth=4.035 m
frame  11   centre=(421, 256)   depth=4.032 m
frame  12   no detection
frame  13   centre=(416, 257)   depth=4.036 m
frame  14   no detection
frame  15   no detection
frame  16   centre=(410, 260)   depth=3.978 m
frame  17   centre=(405, 258)   depth=3.920 m
frame  18   centre=(405, 257)   depth=3.882 m
frame  19   centre=(405, 254)   depth=3.789 m
frame  20   centre=(403, 253)   depth=3.703 m
frame  21   centre=(401, 250)   depth=3.614 m
frame  22   centre=(398, 246)   depth=3.485 m
frame

<Figure size 1100x700 with 2 Axes>

<Figure size 1300x500 with 3 Axes>

<Figure size 1300x500 with 3 Axes>

<Figure size 1300x500 with 3 Axes>

<Figure size 1300x500 with 3 Axes>

# Q2. Object tracking                                                           5 points
  Perform tracking of swinging load on test_swingingload.avi. You should display the trajectory of load as it swings each frame as shown in Fig. below. 

![image.png](attachment:image.png)

Ref: Use help from Ultralytics - https://docs.ultralytics.com/guides/instance-segmentation-and-tracking 

In [18]:
# ============================================================
# Q2 - OBJECT TRACKING
# Track the swinging load, draw its trajectory on every frame,
# and plot the trajectory in the XY and XZ planes.
# ============================================================

import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pyrealsense2 as rs
from ultralytics import YOLO

SESSION = Path("bin/ajit_test_data")
VIDEO   = SESSION / "color_video/color_video.avi"
MODEL   = "runs/q1_model/weights/best.pt"   # trained in Q1
OUTPUT  = "runs/q2_tracking.mp4"

FPS   = 30.0    # L515 recording rate
PATCH = 5       # half-size of the depth patch around the box centre [px]
DEPTH_SCALE = json.loads((SESSION / "depth_scale_json/depth_scale.json").read_text())["depth_scale"]

d = json.loads((SESSION / "depth_scale_json/color_intrinsics.json").read_text())

intr = rs.intrinsics()
intr.width, intr.height = d["width"], d["height"]
intr.fx, intr.fy = d["fx"], d["fy"]
intr.ppx, intr.ppy = d["ppx"], d["ppy"]
intr.model = rs.distortion.brown_conrady
intr.coeffs = d["coeffs"]

def depth_at(frame_idx, u, v):
    """Median depth [m] in a small patch around pixel (u, v), or None if there's no valid reading."""
    path = SESSION / f"depth_raw/depth_{frame_idx:04d}_raw.npy"
    if not path.exists():
        return None
    patch = np.load(path)[max(v - PATCH, 0):v + PATCH + 1, max(u - PATCH, 0):u + PATCH + 1]
    valid = patch[patch > 0]                      # 0 = sensor saw nothing
    return np.median(valid) * DEPTH_SCALE if valid.size else None


# --- output video, same size as the input ---
cap = cv2.VideoCapture(str(VIDEO))
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
writer = cv2.VideoWriter(OUTPUT, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (W, H))

# --- tracking ---
model = YOLO(MODEL)
results = model.track(str(VIDEO), stream=True, verbose=False)

trail  = []   # pixel centres (u, v), for drawing
points = []   # 3D positions (X, Y, Z) in camera frame [m]

for i, result in enumerate(results):
    frame = result.plot()                         # box + confidence

    if len(result.boxes):
        u, v = result.boxes.xywh[0, :2].int().tolist()   # centre of the (only) load
        trail.append((u, v))

        Z = depth_at(i, u, v)
        if Z is not None:
            points.append(rs.rs2_deproject_pixel_to_point(intr, [u, v], Z))

    cv2.polylines(frame, [np.array(trail, np.int32)], False, (0, 255, 0), 2)
    writer.write(frame)

writer.release()
print(f"wrote {OUTPUT} - {len(trail)} detections, depth valid in {len(points)}")

# --- plots: XY (camera view) and XZ (top view) ---
X, Y, Z = np.array(points).T

fig, (ax_xy, ax_xz) = plt.subplots(1, 2, figsize=(11, 5), layout="constrained")

ax_xy.plot(X, Y)
ax_xy.set(xlabel="X [m]", ylabel="Y [m]", title="XY plane (camera view)")
ax_xy.invert_yaxis()                              # image Y points down
ax_xy.grid(alpha=0.3)

ax_xz.plot(X, Z)
ax_xz.set(xlabel="X [m]", ylabel="Z [m]", title="XZ plane (top view)")
ax_xz.grid(alpha=0.3)

FIG_DIR = Path("q2_figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)       # creates it if missing, no error if it exists

fig.savefig(FIG_DIR / "trajectory_xy_xz.png", dpi=200)
plt.show()

wrote runs/q2_tracking.mp4 - 290 detections, depth valid in 275


<Figure size 1100x500 with 2 Axes>

# Q3. Multi object segmentation, tracking and counting:                         10  points

Explore different data-set available freely on internet (https://docs.ultralytics.com/datasets ). Get familiar with them. Then choose a data set and corresponding class of your choice (e.g. CICAR-10 dataset/COCO data set or KITTI data set / person). Then make a video such that multiple instances of objects are in the scene (e.g. 5 people are the scene). Perform 
(a) instance segmentation, 
(b) counting  - total number of object in scene
(c) and tracking of individual instance of object
(d) draw a region - and count the object in that region

In [20]:
# ============================================================
# Q3 - MULTI-OBJECT SEGMENTATION, TRACKING AND COUNTING
# COCO-pretrained YOLO11 segmentation model, class "person".
# (a) instance segmentation   (b) count of objects in the scene
# (c) tracking of each instance   (d) count inside a drawn region
# ============================================================

from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

MODEL_PATH = "yolo11n-seg.pt"           # COCO-pretrained segmentation model, downloads on first use
VIDEO_PATH = "bin/track_30s.mp4"   # swap in our own multi-person clip
OUTPUT_PATH = "runs/q3_counting.mp4"
PERSON = 0                              # COCO class id of "person"
IMGSZ = 1280                            # people are small in a 640x360 clip; 640 finds a third of them

# Region of interest as fractions of (width, height), so it survives a change of video
REGION = np.array([[0.30, 0.35], [0.70, 0.35], [0.70, 1.00], [0.30, 1.00]])

YELLOW, FONT = (0, 255, 255), cv2.FONT_HERSHEY_SIMPLEX

# ------------------------------------------------------------
# 2. SEGMENT, TRACK AND COUNT, FRAME BY FRAME
# ------------------------------------------------------------

model = YOLO(MODEL_PATH)
fps = cv2.VideoCapture(VIDEO_PATH).get(cv2.CAP_PROP_FPS) or 30.0

counts = []          # (people in scene, people in region) per frame
people_seen = set()  # every track id the tracker handed out = people seen in the whole clip
writer = None

for result in model.track(source=VIDEO_PATH, classes=[PERSON], imgsz=IMGSZ,
                          persist=True, stream=True, verbose=False):
    # (a) instance masks + (c) one colour and id per tracked person.
    # conf=False and thin lines keep a crowded frame readable.
    frame = result.plot(conf=False, line_width=1)

    if writer is None:          # first frame fixes the video size and the region in pixels
        height, width = frame.shape[:2]
        region_px = (REGION * [width, height]).astype(np.int32)
        writer = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

    in_scene = len(result.boxes)                    # (b) how many people this frame holds
    centres = result.boxes.xywh.cpu().numpy()[:, :2]

    # (d) a person counts as "in the region" when their box centre falls inside the polygon
    in_region = sum(cv2.pointPolygonTest(region_px, (float(u), float(v)), False) >= 0 for u, v in centres)
    counts.append((in_scene, in_region))

    if result.boxes.id is not None:
        people_seen.update(result.boxes.id.int().cpu().tolist())

    cv2.polylines(frame, [region_px], isClosed=True, color=YELLOW, thickness=2)
    cv2.putText(frame, f"in scene:  {in_scene}", (10, height - 40), FONT, 0.8, YELLOW, 2)
    cv2.putText(frame, f"in region: {in_region}", (10, height - 12), FONT, 0.8, YELLOW, 2)
    writer.write(frame)

writer.release()

# ------------------------------------------------------------
# 3. SUMMARY
# ------------------------------------------------------------

counts = np.array(counts)
print(f"wrote {OUTPUT_PATH}")
print(f"{len(counts)} frames, {len(people_seen)} distinct people tracked over the clip")
print(f"in scene:  mean {counts[:, 0].mean():.1f}, max {counts[:, 0].max()}")
print(f"in region: mean {counts[:, 1].mean():.1f}, max {counts[:, 1].max()}")

FIG_DIR = Path("q3_figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

fig = plt.figure(figsize=(9, 4))
plt.plot(counts[:, 0], label="in scene")
plt.plot(counts[:, 1], label="in region")
plt.xlabel("frame"); plt.ylabel("people"); plt.grid(alpha=0.3); plt.legend()
plt.title("Q3 - people counted per frame")
plt.tight_layout()

fig.savefig(FIG_DIR / "people_per_frame.png", dpi=200)   # save
plt.show()                                               # then show

wrote runs/q3_counting.mp4
750 frames, 160 distinct people tracked over the clip
in scene:  mean 9.5, max 18
in region: mean 1.4, max 5


<Figure size 900x400 with 1 Axes>


# Q4. Classification                                                        10  marks
Given the data-set fault-classification-class. Perform the following
(a) Develop ML model to determine the state of health of asset (in this case motor).
(b) Show, train and val loss, confusion matrix and classification report
(c) Consider the one in the class as baseline and compare your model with it. Summarize in terms of
model parameter, inference time, accuracy, precision, recall, F1 score

In [21]:

# ============================================================
# Q4 - MOTOR FAULT CLASSIFICATION
# ============================================================

import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from tensorflow.keras import layers, regularizers

# ============================================================
# 1. REPRODUCIBILITY AND CONFIGURATION
# ============================================================

SEED = 120
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
n_row = 60
n_col = 175
image_size = (n_row, n_col)
# A batch of 32 gives 45 weight updates per epoch. With the batch of 8 used
# before, the first epoch already contained 180 updates, which was enough to
# solve this easy two-class problem, so the accuracy curve was flat at 1.0
# from epoch 1 and showed nothing.
batch_size = 32
epochs = 30

# The proposed CNN converges much faster than the baseline, so it gets a
# smaller learning rate. Both models still see the same data and epochs.
PROPOSED_LEARNING_RATE = 5e-5
BASELINE_LEARNING_RATE = 1e-4

training_directory = "fault-classification-class/training"
testing_directory = "fault-classification-class/testing"

# All figures are saved automatically in this folder.
FIGURE_DIRECTORY = "q4_figures"
FIGURE_DPI = 300

# One colour per model, reused in every comparison plot.
PROPOSED_COLOR = "#2A6FBB"
BASELINE_COLOR = "#E07A1F"
os.makedirs(FIGURE_DIRECTORY, exist_ok=True)


def save_figure(figure, filename):
    output_path = os.path.join(FIGURE_DIRECTORY, filename)
    figure.savefig(
        output_path,
        dpi=FIGURE_DPI,
        bbox_inches="tight",
        facecolor="white"
    )
    print(f"Figure saved: {output_path}")


# ============================================================
# 2. DATASET LOADING
# ============================================================

train_ds = tf.keras.utils.image_dataset_from_directory(
    training_directory,
    labels="inferred",
    label_mode="int",
    image_size=image_size,
    batch_size=batch_size,
    shuffle=True,
    seed=SEED,
    validation_split=0.20,
    subset="training"
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    training_directory,
    labels="inferred",
    label_mode="int",
    image_size=image_size,
    batch_size=batch_size,
    shuffle=True,
    seed=SEED,
    validation_split=0.20,
    subset="validation"
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    testing_directory,
    labels="inferred",
    label_mode="int",
    image_size=image_size,
    batch_size=batch_size,
    shuffle=False
)
class_names = train_ds.class_names
num_classes = len(class_names)
print("\nClasses found:")
for class_index, class_name in enumerate(class_names):
    print(f"{class_index} = {class_name}")
print(f"\nNumber of classes: {num_classes}")

# ============================================================
# 3. DATASET EXPLORATION
# ============================================================


def get_labels(dataset):
    """Extract all labels from a TensorFlow dataset."""
    return np.concatenate([
        labels.numpy().reshape(-1)
        for _, labels in dataset
    ])


def print_class_distribution(dataset, dataset_name):
    """Print number and percentage of samples for every class."""
    labels = get_labels(dataset)
    total_samples = len(labels)
    print(f"\n{dataset_name} distribution")
    print("-" * 40)
    for class_index, class_name in enumerate(class_names):
        count = int(np.sum(labels == class_index))
        percentage = 100 * count / total_samples
        print(
            f"{class_index} - {class_name}: "
            f"{count} samples ({percentage:.2f}%)"
        )
    print(f"Total: {total_samples} samples")


print_class_distribution(train_ds, "Training set")
print_class_distribution(val_ds, "Validation set")
print_class_distribution(test_ds, "Test set")

# ============================================================
# 4. DISPLAY EXAMPLE IMAGES
# ============================================================

for images, labels in train_ds.take(1):
    number_to_show = min(6, images.shape[0])
    fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(13, 4.2))
    axes = axes.flatten()
    for i in range(len(axes)):
        if i < number_to_show:
            image = images[i].numpy().astype("uint8")
            label_index = int(labels[i].numpy())
            axes[i].imshow(image)
            axes[i].set_title(
                class_names[label_index],
                fontsize=10,
                fontweight="bold",
                pad=8,
            )
        axes[i].axis("off")
    fig.suptitle("Motor Fault Dataset - Training Examples", fontsize=14)
    # The samples are wide strips, so the spacing is set explicitly:
    # tight_layout cannot reclaim the slack left by the fixed image aspect.
    fig.subplots_adjust(top=0.85, hspace=0.35, wspace=0.1)
    save_figure(fig, "01_training_examples.png")
    plt.show()

# ============================================================
# 5. INPUT NORMALIZATION
# ============================================================

normalizer = layers.Normalization(axis=-1)
normalizer.adapt(train_ds.map(lambda images, labels: images))

# ============================================================
# 7. CNN MODEL
# ============================================================

# Fully convolutional: the three convolution blocks extract the texture that
# separates the two classes, global average pooling reduces each feature map to
# one number, and the softmax reads those 16 numbers directly. The Dense(16)
# and Dropout layers that used to sit between them were removed: they held 14%
# of the weights and dropping them raised the test accuracy and lowered the
# test loss.
model = tf.keras.Sequential([
    layers.Input(shape=(n_row, n_col, 3)),
    normalizer,
    layers.Conv2D(filters=4, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Conv2D(filters=8, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Conv2D(filters=16, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling2D(),
    layers.Dense(units=num_classes, activation="softmax")
], name='class_proposed')
model.summary()
print(f"\nTotal model parameters: {model.count_params():,}")

# ============================================================
# 8. MODEL COMPILATION
# ============================================================


class TrainSetEvaluation(tf.keras.callbacks.Callback):
    """Re-measure the training set at the end of each epoch.

    Keras reports the training accuracy as a running average taken across the
    epoch, while the model is still improving. Validation is measured once at
    the end of the epoch with the final weights. That difference alone made the
    training curve sit below the validation curve. This callback measures the
    training set the same way validation is measured, so the two curves can be
    compared directly.
    """

    def __init__(self, dataset):
        super().__init__()
        self.dataset = dataset

    def on_epoch_end(self, epoch, logs=None):
        loss, accuracy = self.model.evaluate(self.dataset, verbose=0)
        logs["train_loss"] = loss
        logs["train_accuracy"] = accuracy


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=PROPOSED_LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

# ============================================================
# 10. MODEL TRAINING
# ============================================================

history = model.fit(
    train_ds,
    epochs=epochs,
    verbose=1,
    validation_data=val_ds,
    callbacks=[TrainSetEvaluation(train_ds)]
)

# ============================================================
# 11. VALIDATION RESULTS
# ============================================================

validation_loss, validation_accuracy = model.evaluate(val_ds, verbose=0)
print("\nValidation results")
print("-" * 40)
print(f"Validation loss:     {validation_loss:.6f}")
print(f"Validation accuracy: {validation_accuracy:.4f}")
print(f"Validation accuracy: {validation_accuracy * 100:.2f}%")

# ============================================================
# 12. TRAINING AND VALIDATION CURVES
# ============================================================

# train_accuracy / train_loss come from TrainSetEvaluation and are measured
# the same way as the validation metrics.
acc = history.history['train_accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['train_loss']
val_loss = history.history['val_loss']
epochs_range = range(1, len(acc) + 1)
loss_figure = plt.figure(figsize=(6.5, 5))
plt.plot(epochs_range, loss, label='Training Loss', linewidth=2)
plt.plot(epochs_range, val_loss, label='Validation Loss', linewidth=2)
plt.legend(loc='upper right')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Proposed CNN – Training and Validation Loss')
plt.grid(alpha=0.25)
plt.tight_layout()
save_figure(loss_figure, "02_proposed_training_loss.png")
plt.show()
accuracy_figure = plt.figure(figsize=(6.5, 5))
plt.plot(epochs_range, acc, label='Training Accuracy', linewidth=2)
plt.plot(epochs_range, val_acc, label='Validation Accuracy', linewidth=2)
plt.legend(loc='lower right')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Proposed CNN – Training and Validation Accuracy')
plt.ylim(0, 1.05)
plt.grid(alpha=0.25)
plt.tight_layout()
save_figure(accuracy_figure, "03_proposed_training_accuracy.png")
plt.show()

# ============================================================
# 13. TEST SET EVALUATION
# ============================================================

# Evaluate the final model on the test set
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
print("\nTest set results")
print("-" * 40)
print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy * 100:.2f}%")

# ============================================================
# 14. TEST SET PREDICTIONS
# ============================================================

# Predicted probabilities for every class
y_probabilities = model.predict(test_ds,verbose=0)
# Convert probabilities to predicted class indices
y_pred = np.argmax(y_probabilities, axis=1)
# Extract true class indices
y_true = np.concatenate([
    labels.numpy()
    for _, labels in test_ds
])
# Verify that labels and predictions have the same length
print(f"\nNumber of test samples: {len(y_true)}")
print(f"Number of predictions: {len(y_pred)}")

# ============================================================
# 15. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))
# Normalize each row by the number of true samples in that class
cm_normalized = confusion_matrix(y_true, y_pred, labels=range(num_classes), normalize="true")
# Absolute confusion matrix
counts_figure = plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    cbar=False,
    annot_kws={"size": 13}
)
plt.title("Test Confusion Matrix – Sample Counts")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
save_figure(counts_figure, "04_proposed_confusion_matrix_counts.png")
plt.show()
# Normalized confusion matrix
normalized_figure = plt.figure(figsize=(7, 6))
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    vmin=0,
    vmax=1,
    xticklabels=class_names,
    yticklabels=class_names,
    cbar=True,
    annot_kws={"size": 13}
)
plt.title("Test Confusion Matrix – Normalized")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
save_figure(normalized_figure, "05_proposed_confusion_matrix_normalized.png")
plt.show()

# ============================================================
# 16. CLASSIFICATION REPORT
# ============================================================

report = classification_report(
    y_true,
    y_pred,
    labels=range(num_classes),
    target_names=class_names,
    digits=4,
    zero_division=0
)
print("\nTest classification report")
print("-" * 60)
print(report)

# ============================================================
# 17. CLASS BASELINE MODEL
# ============================================================

# Reset the seed before building the baseline, so that its weight
# initialisation is reproducible and does not depend on the random state left
# behind by the training of the proposed model.
tf.keras.utils.set_random_seed(SEED)

# A separate normalization layer is used so that the two models
# remain completely independent.
baseline_normalizer = layers.Normalization(axis=-1)
baseline_normalizer.adapt(train_ds.map(lambda images, labels: images))
baseline_model = tf.keras.Sequential([
    layers.Input(shape=(n_row, n_col, 3)),
    baseline_normalizer,
    layers.Conv2D(filters=4, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(filters=8, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(filters=8, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(filters=16, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(),
    layers.BatchNormalization(),
    layers.Flatten(),
    layers.Dense(units=16, activation="relu", activity_regularizer=regularizers.L2(0.01)),
    layers.Dense(units=16, activation="relu", activity_regularizer=regularizers.L2(0.01)),
    layers.Dense(units=2, activation="relu"),
    layers.Dense(units=64, activation="relu", activity_regularizer=regularizers.L2(0.01)),
    layers.Dense(units=64, activation="relu", activity_regularizer=regularizers.L2(0.01)),
    layers.Dropout(rate=0.2),
    layers.Dense(units=num_classes, activation="softmax")
], name="class_baseline")
baseline_model.summary()
print(
    f"\nTotal baseline model parameters: "
    f"{baseline_model.count_params():,}"
)

# ============================================================
# 18. BASELINE COMPILATION AND TRAINING
# ============================================================

baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=BASELINE_LEARNING_RATE
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)
baseline_epochs = epochs
baseline_history = baseline_model.fit(
    train_ds,
    epochs=baseline_epochs,
    verbose=1,
    validation_data=val_ds,
    callbacks=[TrainSetEvaluation(train_ds)]
)

# ============================================================
# 19. BASELINE VALIDATION RESULTS
# ============================================================

baseline_validation_loss, baseline_validation_accuracy = baseline_model.evaluate(val_ds, verbose=0)
print("\nBaseline validation results")
print("-" * 40)
print(f"Validation loss:     {baseline_validation_loss:.6f}")
print(f"Validation accuracy: {baseline_validation_accuracy:.4f}")
print(f"Validation accuracy: {baseline_validation_accuracy * 100:.2f}%")

# ============================================================
# 20. TRAINING CURVES COMPARISON
# ============================================================

proposed_epoch_range = range(1,len(history.history["loss"]) + 1)
baseline_epoch_range = range(1,len(baseline_history.history["loss"]) + 1)
# Colour identifies the model and the line style the split, so the two
# models stay easy to tell apart.
curves = [
    ("Proposed", history, PROPOSED_COLOR, proposed_epoch_range),
    ("Baseline", baseline_history, BASELINE_COLOR, baseline_epoch_range),
]

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(13, 5))

for metric, axis, title, y_label in [
    ("loss", axes[0], "Loss Comparison", "Loss"),
    ("accuracy", axes[1], "Accuracy Comparison", "Accuracy"),
]:
    for model_name, model_history, color, epoch_range in curves:
        axis.plot(
            epoch_range,
            model_history.history[f"train_{metric}"],
            linestyle="--",
            linewidth=2,
            color=color,
            label=f"{model_name} - Training",
        )
        axis.plot(
            epoch_range,
            model_history.history[f"val_{metric}"],
            linewidth=2,
            color=color,
            label=f"{model_name} - Validation",
        )
    axis.set_title(title)
    axis.set_xlabel("Epoch")
    axis.set_ylabel(y_label)
    axis.grid(alpha=0.25)
    axis.legend()

axes[1].set_ylim(0, 1.05)
plt.tight_layout()
save_figure(fig, "06_training_curves_comparison.png")
plt.show()

# ============================================================
# 21. PREPARE TEST DATA FOR FAIR TIMING
# ============================================================

# Test images are loaded into memory so that inference timing
# does not include disk reading time.
test_image_batches = []
test_label_batches = []
for images, labels in test_ds:
    test_image_batches.append(images.numpy())
    test_label_batches.append(labels.numpy())
test_images = np.concatenate(
    test_image_batches,
    axis=0
)
y_true = np.concatenate(
    test_label_batches,
    axis=0
)
print(f"\nNumber of test images: {len(test_images)}")

# ============================================================
# 22. MODEL EVALUATION FUNCTION
# ============================================================

# function that takes a model already trained and
# - evaluated it on the test set
# - measure the inference time
# - return useful information for comparison with other models
def evaluate_classifier(classifier, model_name, test_images, y_true, test_dataset, batch_size, timing_repeats=5,):
    test_loss, _ = classifier.evaluate(test_dataset, verbose=0)
    # Full warm-up run to avoid including TensorFlow tracing time.
    classifier.predict(test_images, batch_size=batch_size, verbose=0)
    elapsed_times = []
    probabilities = None
    for _ in range(timing_repeats):
        start_time = time.perf_counter()
        probabilities = classifier.predict(test_images,batch_size=batch_size, verbose=0)
        end_time = time.perf_counter()
        elapsed_times.append(end_time - start_time)
    y_pred = np.argmax(probabilities, axis=1)
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1_score, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=range(num_classes),
            average="macro",
            zero_division=0
        )
    )
    mean_inference_seconds = np.mean(elapsed_times)
    inference_ms_per_image = (mean_inference_seconds * 1000 / len(test_images))
    results = {
        "Model": model_name,
        "Parameters": classifier.count_params(),
        "Test loss": test_loss,
        "Accuracy": accuracy,
        "Precision (macro)": precision,
        "Recall (macro)": recall,
        "F1 score (macro)": f1_score,
        "Inference (ms/image)": inference_ms_per_image
    }
    report = classification_report(
        y_true,
        y_pred,
        labels=range(num_classes),
        target_names=class_names,
        digits=4,
        zero_division=0
    )
    return results, y_pred, report


# ============================================================
# 23. EVALUATE BOTH MODELS
# ============================================================

proposed_results, proposed_predictions, proposed_report = (
    evaluate_classifier(
        classifier=model,
        model_name="Proposed CNN",
        test_images=test_images,
        y_true=y_true,
        test_dataset=test_ds,
        batch_size=batch_size
    )
)
baseline_results, baseline_predictions, baseline_report = (
    evaluate_classifier(
        classifier=baseline_model,
        model_name="Class baseline",
        test_images=test_images,
        y_true=y_true,
        test_dataset=test_ds,
        batch_size=batch_size
    )
)

# ============================================================
# 24. CLASSIFICATION REPORTS
# ============================================================

print("\nProposed CNN classification report")
print("-" * 60)
print(proposed_report)
print("\nClass baseline classification report")
print("-" * 60)
print(baseline_report)

# ============================================================
# 25. NUMERICAL COMPARISON TABLE
# ============================================================

comparison = pd.DataFrame([
    proposed_results,
    baseline_results
]).set_index("Model")
comparison_display = comparison.copy()
comparison_display["Parameters"] = (
    comparison_display["Parameters"]
    .map(lambda value: f"{int(value):,}")
)
comparison_display["Test loss"] = (
    comparison_display["Test loss"]
    .map(lambda value: f"{value:.4f}")
)
for column in [
    "Accuracy",
    "Precision (macro)",
    "Recall (macro)",
    "F1 score (macro)"
]:
    comparison_display[column] = (
        comparison_display[column]
        .map(lambda value: f"{value:.4f}")
    )
comparison_display["Inference (ms/image)"] = (
    comparison_display["Inference (ms/image)"]
    .map(lambda value: f"{value:.4f}")
)
print("\nModel comparison")
print("-" * 100)
print(comparison_display.to_string())

# ============================================================
# 26. KPI COMPARISON PLOTS
# ============================================================

model_colors = [PROPOSED_COLOR, BASELINE_COLOR]
metric_columns = [
    "Accuracy",
    "Precision (macro)",
    "Recall (macro)",
    "F1 score (macro)"
]


def metric_axis_limits(values):
    """Zoom the axis on the bars, leaving room for the value labels."""
    lowest, highest = float(values.min()), float(values.max())
    margin = max(highest - lowest, 0.01) * 0.5
    return max(0.0, lowest - margin), highest + 2 * margin


def label_bars(axis, format_value):
    """Write the value on top of every bar of an axis."""
    for container in axis.containers:
        axis.bar_label(
            container,
            labels=[format_value(bar.get_height()) for bar in container],
            padding=3,
            fontsize=8
        )


fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(18, 5))

# Classification metrics. Both models score close to 1, so the axis is
# zoomed on the region that actually contains the bars.
metrics_table = comparison[metric_columns]
metrics_table.T.plot(kind="bar", ax=axes[0], color=model_colors, width=0.7)
label_bars(axes[0], lambda value: f"{value:.4f}")
axes[0].set_title("Test Classification Metrics")
axes[0].set_xlabel("")
axes[0].set_ylabel("Score")
axes[0].set_ylim(*metric_axis_limits(metrics_table.values))
axes[0].tick_params(axis="x", rotation=15)
axes[0].grid(axis="y", alpha=0.25)
axes[0].set_axisbelow(True)
axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2, frameon=False)

# Number of parameters and inference time.
for axis, column, title, y_label, format_value in [
    (axes[1], "Parameters", "Model Parameters", "Number of parameters",
     lambda value: f"{value:,.0f}"),
    (axes[2], "Inference (ms/image)", f"Inference Time - Batch Size {batch_size}",
     "Milliseconds per image", lambda value: f"{value:.4f}"),
]:
    axis.bar(comparison.index, comparison[column], color=model_colors, width=0.55)
    label_bars(axis, format_value)
    axis.set_title(title)
    axis.set_ylabel(y_label)
    axis.set_ylim(0, comparison[column].max() * 1.15)
    axis.tick_params(axis="x", rotation=10)
    axis.grid(axis="y", alpha=0.25)
    axis.set_axisbelow(True)

plt.tight_layout()
save_figure(fig, "07_kpi_comparison.png")
plt.show()

# ============================================================
# 27. CONFUSION MATRIX COMPARISON
# ============================================================

prediction_sets = [
    (proposed_predictions, "Proposed CNN"),
    (baseline_predictions, "Class baseline")
]
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(13, 5))
for column_index, (predictions, model_name) in enumerate(prediction_sets):
    cm_normalized = confusion_matrix(
        y_true,
        predictions,
        labels=range(num_classes),
        normalize="true"
    )
    sns.heatmap(
        cm_normalized,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        vmin=0,
        vmax=1,
        xticklabels=class_names,
        yticklabels=class_names,
        cbar=(column_index == 1),
        annot_kws={"size": 12},
        ax=axes[column_index]
    )
    axes[column_index].set_title(f"{model_name} - Normalized")
    axes[column_index].set_xlabel("Predicted Label")
    axes[column_index].set_ylabel("True Label")
plt.tight_layout()
save_figure(fig, "08_confusion_matrix_comparison.png")
plt.show()

# ============================================================
# 28. AUTOMATIC SUMMARY
# ============================================================

accuracy_difference = (proposed_results["Accuracy"] - baseline_results["Accuracy"]) * 100
f1_difference = (proposed_results["F1 score (macro)"] - baseline_results["F1 score (macro)"]) * 100
parameter_reduction = (
    (baseline_results["Parameters"] - proposed_results["Parameters"])
    / baseline_results["Parameters"]
    * 100
)
# Parameter count does not predict latency. The proposed model holds 82%
# fewer weights but keeps wider feature maps and is measured as the slower of
# the two, so the ratio below comes out under 1. It is printed as a ratio with
# an explicit direction rather than as a "speed-up", which would read as a
# gain while describing a loss.
inference_ratio = (
    baseline_results["Inference (ms/image)"]
    / proposed_results["Inference (ms/image)"]
)
print("\nComparison summary")
print("-" * 60)
print(
    "Accuracy difference "
    f"(proposed - baseline): "
    f"{accuracy_difference:+.2f} percentage points"
)
print(
    "Macro F1 difference "
    f"(proposed - baseline): "
    f"{f1_difference:+.2f} percentage points"
)
print(
    "Parameter reduction of the proposed model: "
    f"{parameter_reduction:+.2f}%"
)
faster_model, slower_model = (
    ("proposed", "baseline") if inference_ratio > 1 else ("baseline", "proposed")
)
print(
    "Inference time ratio "
    f"(baseline time / proposed time): {inference_ratio:.2f}x "
    f"-> the {faster_model} model is the faster of the two"
)

Found 1800 files belonging to 2 classes.
Using 1440 files for training.
Found 1800 files belonging to 2 classes.
Using 360 files for validation.
Found 200 files belonging to 2 classes.

Classes found:
0 = 00_15
1 = 10_15

Number of classes: 2

Training set distribution
----------------------------------------
0 - 00_15: 716 samples (49.72%)
1 - 10_15: 724 samples (50.28%)
Total: 1440 samples

Validation set distribution
----------------------------------------
0 - 00_15: 184 samples (51.11%)
1 - 10_15: 176 samples (48.89%)
Total: 360 samples

Test set distribution
----------------------------------------
0 - 00_15: 100 samples (50.00%)
1 - 10_15: 100 samples (50.00%)
Total: 200 samples
Figure saved: q4_figures/01_training_examples.png


<Figure size 1300x420 with 6 Axes>

Model: "class_proposed"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ normalization_2 (Normalization) │ (None, 60, 175, 3)     │             7 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 60, 175, 4)     │           112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 30, 87, 4)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 30, 87, 8)      │           296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 15, 43, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 15, 43, 16)     │         1,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 7, 21, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 7, 21, 16)      │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 16)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,681 (6.57 KB)

 Trainable params: 1,642 (6.41 KB)

 Non-trainable params: 39 (160.00 B)


Total model parameters: 1,681
Epoch 1/30


RuntimeError: pybind11::error_already_set: MISMATCH of original and normalized active exception types: ORIGINAL InternalError REPLACED BY KeyboardInterrupt: <EMPTY MESSAGE>

At:
  /home/havard/hand_in/MAS512_Assignment_1/.venv/lib/python3.10/site-packages/tensorflow/python/framework/errors_impl.py(462): __init__
  /home/havard/hand_in/MAS512_Assignment_1/.venv/lib/python3.10/site-packages/tensorflow/python/eager/execute.py(53): quick_execute
  /home/havard/hand_in/MAS512_Assignment_1/.venv/lib/python3.10/site-packages/tensorflow/python/eager/context.py(1688): call_function
  /home/havard/hand_in/MAS512_Assignment_1/.venv/lib/python3.10/site-packages/tensorflow/python/eager/polymorphic_function/atomic_function.py(251): call_flat
  /home/havard/hand_in/MAS512_Assignment_1/.venv/lib/python3.10/site-packages/tensorflow/python/eager/polymorphic_function/atomic_function.py(216): call_preflattened
  /home/havard/hand_in/MAS512_Assignment_1/.venv/lib/python3.10/site-packages/tensorflow/python/

# Q5. Classification                                                        20  
Download CFAR-10 dataset (https://www.tensorflow.org/datasets/catalog/cifar10). Perform the classification task on this dataset 
(a) Use model fusion (use two or models to extract feature and combine them), 
(b) Use transfer learning
(c) Use finetuning
(d) Compare different methods (a) - (c) in terms of typical KPIs often used for classification

In [ ]:
# ============================================================
# 1. IMPORT LIBRARIES AND SETTINGS
# ============================================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn
import tensorflow as tf

from tensorflow.keras.applications import MobileNetV2, EfficientNetB0
from tensorflow.keras.datasets import cifar10

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
)

print("Python:", sys.executable)
print("TensorFlow:", tf.__version__)
print("Detected GPUs:", tf.config.list_physical_devices("GPU"))

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print("GPU memory growth enabled")
    except RuntimeError as error:
        print(error)

BATCH_SIZE = 128            # head training on cached features
FINE_TUNING_BATCH_SIZE = 64  # end-to-end training
HEAD_EPOCHS = 50
FINE_TUNING_EPOCHS = 25
VALIDATION_SPLIT = 0.20
NUMBER_OF_CLASSES = 10
MODEL_IMAGE_SIZE = (96, 96)
FINE_TUNE_LAYERS = 40
HEAD_DROPOUT = 0.30
RANDOM_SEED = 42

# Learning rates: the fusion head sees 2560 features and the transfer head
# 1280, so they are tuned separately. Fine-tuning updates the backbone and
# needs a much smaller rate (1e-4 was unstable, 3e-5 noisy).
FUSION_LEARNING_RATE = 5e-5
TRANSFER_LEARNING_RATE = 1e-4
FINE_TUNING_LEARNING_RATE = 1e-5

np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)

# All figures are saved automatically in this folder.
FIGURE_DIRECTORY = "q5_figures"
os.makedirs(FIGURE_DIRECTORY, exist_ok=True)

# One colour per method, reused in every comparison plot.
MODEL_COLORS = {
    "Model Fusion": "#2A6FBB",
    "Transfer Learning": "#E07A1F",
    "Fine-tuning": "#2E8B57",
}

TRAINING_COLOR = "#2A6FBB"
VALIDATION_COLOR = "#E07A1F"
BEST_EPOCH_COLOR = "#C0392B"


def save_figure(figure, model_name, suffix):
    file_name = model_name.lower().replace(" ", "_").replace("-", "_")
    output_path = os.path.join(FIGURE_DIRECTORY, f"{file_name}_{suffix}.png")
    figure.savefig(output_path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"Figure saved: {output_path}")


# ============================================================
# 2. LOAD CIFAR-10
# ============================================================

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

y_train = y_train.reshape(-1)
y_test = y_test.reshape(-1)

# Keep pixels in [0, 255]; each model applies its own preprocessing.
x_train = x_train.astype("float32")
x_test = x_test.astype("float32")

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

print("Train images shape:", x_train.shape)
print("Test images shape:", x_test.shape)

plt.figure(figsize=(10, 6))
sample_generator = np.random.default_rng(RANDOM_SEED)
for i, index in enumerate(sample_generator.choice(len(x_train), 12, replace=False)):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_train[index].astype("uint8"))
    plt.title(class_names[y_train[index]])
    plt.axis("off")
plt.tight_layout()
plt.show()

# Explicit split: the last 20% of the training set is used for validation.
validation_start = int(len(x_train) * (1.0 - VALIDATION_SPLIT))
x_train_fit, y_train_fit = x_train[:validation_start], y_train[:validation_start]
x_validation, y_validation = x_train[validation_start:], y_train[validation_start:]

print("Training set:", x_train_fit.shape)
print("Validation set:", x_validation.shape)
print("Test set:", x_test.shape)


# ============================================================
# 3. COMMON FUNCTIONS
# ============================================================


class TrainSetEvaluation(tf.keras.callbacks.Callback):
    """Re-measure the training set at the end of each epoch.

    Keras reports the training metrics as a running average taken across the
    epoch, with Dropout active and, for fine-tuning, on augmented images.
    Validation is measured once at the end of the epoch on clean images with
    Dropout off. That difference alone makes the training curve sit *below*
    the validation curve, which reads as if the model generalised better than
    it fits. This callback measures the training set the same way validation
    is measured, so the two curves are directly comparable.
    """

    def __init__(self, x, y, batch_size=256):
        super().__init__()
        self.x, self.y, self.batch_size = x, y, batch_size

    def on_epoch_end(self, epoch, logs=None):
        loss, accuracy = self.model.evaluate(
            self.x, self.y, batch_size=self.batch_size, verbose=0
        )
        logs["train_loss"] = loss
        logs["train_accuracy"] = accuracy


def make_data_augmentation(name):
    '''Light transformations, applied only while the model is training.'''
    return tf.keras.Sequential(
        [
            tf.keras.layers.RandomFlip("horizontal", seed=RANDOM_SEED),
            tf.keras.layers.RandomTranslation(
                height_factor=0.04, width_factor=0.04,
                fill_mode="reflect", seed=RANDOM_SEED,
            ),
            tf.keras.layers.RandomRotation(
                factor=0.02, fill_mode="reflect", seed=RANDOM_SEED,
            ),
            tf.keras.layers.RandomZoom(
                height_factor=0.05, width_factor=0.05,
                fill_mode="reflect", seed=RANDOM_SEED,
            ),
        ],
        name=name,
    )


def print_model_parameters(model):
    total_params = model.count_params()
    trainable_params = sum(
        tf.keras.backend.count_params(weight) for weight in model.trainable_weights
    )
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Trainable ratio: {trainable_params / total_params * 100:.2f}%")


def make_classifier_head(feature_dimension, name):
    """Dropout straight onto the pooled features, then a linear softmax.

    A Dense(256) layer here held 97% of the trainable weights and memorised
    the frozen features: validation loss stopped improving while training loss
    kept falling. Removing it lowered the validation loss and closed the gap.
    Fusion and transfer learning share this head, so the difference between
    them is attributable to two feature extractors versus one.
    """
    return tf.keras.Sequential(
        [
            tf.keras.Input(shape=(feature_dimension,)),
            tf.keras.layers.Dropout(HEAD_DROPOUT),
            tf.keras.layers.Dense(NUMBER_OF_CLASSES, activation="softmax"),
        ],
        name=name,
    )


def extract_features(extractor, images, label):
    """Run the frozen backbones once instead of on every epoch.

    The backbones are frozen and these two paths use no augmentation, so the
    features are deterministic: caching them is only a speed-up, not a change
    of model. Verified by reassembling the head onto the full network, which
    reproduced the same test accuracy to within one image in ten thousand.
    """
    start = time.perf_counter()
    features = extractor.predict(images, batch_size=256, verbose=0)
    print(f"  {label}: {features.shape} in {time.perf_counter() - start:.0f}s")
    return features


def plot_training_history(history, model_name):
    epochs = np.arange(1, len(history["val_loss"]) + 1)
    best_epoch_index = int(np.argmin(history["val_loss"]))
    best_epoch = int(epochs[best_epoch_index])

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    for axis, (metric, y_label, title) in zip(axes, [
        ("accuracy", "Accuracy", "Training and Validation Accuracy"),
        ("loss", "Loss", "Training and Validation Loss"),
    ]):
        # train_* comes from TrainSetEvaluation and is measured the same way
        # as the validation metrics.
        axis.plot(epochs, history[f"train_{metric}"], color=TRAINING_COLOR,
                  marker="o", markersize=3, linewidth=2.2,
                  label=f"Training {y_label.lower()}")
        axis.plot(epochs, history[f"val_{metric}"], color=VALIDATION_COLOR,
                  marker="o", markersize=3, linewidth=2.2,
                  label=f"Validation {y_label.lower()}")
        axis.axvline(best_epoch, color=BEST_EPOCH_COLOR, linestyle=":",
                     linewidth=1.6, label=f"Best epoch: {best_epoch}")
        axis.set_xlabel("Epoch")
        axis.set_ylabel(y_label)
        axis.set_title(title)
        axis.grid(alpha=0.25)
        axis.set_axisbelow(True)
        axis.legend(frameon=True)

    fig.suptitle(
        f"{model_name} - Learning Curves\n"
        f"Best epoch by validation loss: {best_epoch} - "
        f"validation accuracy: {history['val_accuracy'][best_epoch_index]:.4f} - "
        f"validation loss: {history['val_loss'][best_epoch_index]:.4f}",
        fontsize=15, fontweight="bold",
    )
    fig.tight_layout()
    save_figure(fig, model_name, "training_history")
    plt.show()


def evaluate_classifier(classifier, model_name, test_images, y_true,
                        training_time, batch_size=128, timing_repeats=3):
    '''Evaluate a classifier and return the KPIs used in the comparison.

    Always called on the full assembled model, so the inference time is
    comparable across the three methods even though two of them trained on
    cached features.
    '''
    test_loss, test_acc = classifier.evaluate(
        test_images, y_true, batch_size=batch_size, verbose=0
    )

    classifier.predict(test_images[:batch_size], batch_size=batch_size, verbose=0)

    # The whole test set is timed in batches. Timing a single image would
    # measure the per-call framework overhead instead of the model itself.
    elapsed_times, probabilities = [], None
    for _ in range(timing_repeats):
        start = time.perf_counter()
        probabilities = classifier.predict(test_images, batch_size=batch_size, verbose=0)
        elapsed_times.append(time.perf_counter() - start)
    inference_ms_per_image = np.mean(elapsed_times) * 1000 / len(test_images)

    y_pred = np.argmax(probabilities, axis=-1)
    macro_precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    macro_recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    print(f"\nModel: {model_name}")
    print(f"Test loss: {test_loss:.4f}")
    print(f"Test accuracy: {test_acc:.4f}")
    print(f"Macro precision: {macro_precision:.4f}")
    print(f"Macro recall: {macro_recall:.4f}")
    print(f"Macro F1 score: {macro_f1:.4f}")
    print(f"Training time: {training_time:.2f} seconds")
    print(f"Inference time: {inference_ms_per_image:.4f} ms/image (batch size {batch_size})")

    print("\nClassification report")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    cm_normalized = cm / cm.sum(axis=1, keepdims=True)

    fig = plt.figure(figsize=(9, 7))
    sn.heatmap(cm_normalized, xticklabels=class_names, yticklabels=class_names,
               annot=True, fmt=".2f", annot_kws={"size": 9}, cmap="Blues", vmin=0, vmax=1)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(f"{model_name} - Normalized Confusion Matrix (test set)")
    plt.tight_layout()
    save_figure(fig, model_name, "confusion_matrix")
    plt.show()

    results = {
        "Model": model_name,
        "Test loss": test_loss,
        "Accuracy": test_acc,
        "Macro precision": macro_precision,
        "Macro recall": macro_recall,
        "Macro F1": macro_f1,
        "Training time (s)": training_time,
        "Inference (ms/image)": inference_ms_per_image,
        "Total parameters": classifier.count_params(),
        "Trainable parameters": sum(
            tf.keras.backend.count_params(w) for w in classifier.trainable_weights
        ),
    }
    return results, y_pred, cm_normalized


def visualize_predictions(model_name, test_images, y_true, y_pred, number_of_samples=12):
    # A fixed generator keeps the same images across runs and across models.
    generator = np.random.default_rng(RANDOM_SEED)
    indices = generator.choice(len(test_images), number_of_samples, replace=False)
    number_of_columns = 4
    number_of_rows = (number_of_samples + number_of_columns - 1) // number_of_columns

    fig = plt.figure(figsize=(15, 3 * number_of_rows))
    for i, index in enumerate(indices):
        plt.subplot(number_of_rows, number_of_columns, i + 1)
        plt.imshow(test_images[index].astype("uint8"))
        is_correct = y_pred[index] == y_true[index]
        title = (f"True: {class_names[y_true[index]]}\n"
                 f"Pred: {class_names[y_pred[index]]}")
        if not is_correct:
            title += "\nMISCLASSIFIED"
        plt.title(title, color="green" if is_correct else "red", fontsize=10)
        plt.axis("off")
    plt.suptitle(f"{model_name} - Sample test predictions", fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    save_figure(fig, model_name, "sample_predictions")
    plt.show()


# ============================================================
# 4. (a) MODEL FUSION
# ============================================================
# Two ImageNet backbones are used as feature extractors. Each is pooled to a
# 1280-d vector, LayerNormalized so neither dominates, and the two are
# concatenated into a single 2560-d fused representation.

fusion_mobile_base = MobileNetV2(weights="imagenet", include_top=False,
                                 input_shape=(*MODEL_IMAGE_SIZE, 3))
fusion_efficientnet_base = EfficientNetB0(weights="imagenet", include_top=False,
                                          input_shape=(*MODEL_IMAGE_SIZE, 3))
fusion_mobile_base.trainable = False
fusion_efficientnet_base.trainable = False

fusion_inputs = tf.keras.Input(shape=(32, 32, 3), name="image")
resized = tf.keras.layers.Resizing(*MODEL_IMAGE_SIZE, interpolation="bilinear",
                                   name="fusion_resize")(fusion_inputs)

# MobileNetV2 expects [-1, 1]; EfficientNetB0 rescales internally from [0, 255].
mobile_inputs = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1.0,
                                          name="mobilenet_preprocessing")(resized)

mobile_features = tf.keras.layers.GlobalAveragePooling2D(name="mobilenet_features")(
    fusion_mobile_base(mobile_inputs, training=False))
efficientnet_features = tf.keras.layers.GlobalAveragePooling2D(name="efficientnet_features")(
    fusion_efficientnet_base(resized, training=False))

mobile_features = tf.keras.layers.LayerNormalization(
    name="mobilenet_feature_normalization")(mobile_features)
efficientnet_features = tf.keras.layers.LayerNormalization(
    name="efficientnet_feature_normalization")(efficientnet_features)

fused_features = tf.keras.layers.Concatenate(name="feature_fusion")(
    [mobile_features, efficientnet_features])

fusion_extractor = tf.keras.Model(fusion_inputs, fused_features, name="fusion_extractor")
# The extractor is frozen end to end. The two backbones already were, but the
# LayerNormalization gammas and betas were not, and they stayed at their
# initial values because only the head is ever fitted. Left trainable they
# added 5,120 weights to the "trainable parameters" reported for fusion,
# 20% of the bar, for weights no optimizer ever touched. LayerNormalization
# keeps no running statistics, so freezing it changes no output.
fusion_extractor.trainable = False

print("Extracting fused features (runs the two backbones once, not once per epoch)")
fusion_train = extract_features(fusion_extractor, x_train_fit, "train")
fusion_validation = extract_features(fusion_extractor, x_validation, "validation")

tf.keras.utils.set_random_seed(RANDOM_SEED)
fusion_head = make_classifier_head(fused_features.shape[-1], "fusion_head")
fusion_head.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FUSION_LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

start_time = time.perf_counter()
history_fusion = fusion_head.fit(
    fusion_train, y_train_fit,
    validation_data=(fusion_validation, y_validation),
    epochs=HEAD_EPOCHS, batch_size=BATCH_SIZE, verbose=0,
    callbacks=[TrainSetEvaluation(fusion_train, y_train_fit)],
).history
training_time_fusion = time.perf_counter() - start_time
print(f"Total training time: {training_time_fusion:.2f} seconds")

# Assemble extractor + head into the complete fusion model, so evaluation and
# inference timing run the real end-to-end network.
model_fusion = tf.keras.Model(fusion_inputs, fusion_head(fused_features),
                              name="mobilenet_efficientnet_fusion")
model_fusion.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FUSION_LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)
print_model_parameters(model_fusion)

plot_training_history(history_fusion, "Model Fusion")
fusion_results, y_pred_fusion, cm_fusion = evaluate_classifier(
    model_fusion, "Model Fusion", x_test, y_test, training_time_fusion)
visualize_predictions("Model Fusion", x_test, y_test, y_pred_fusion)


# ============================================================
# 5. (b) TRANSFER LEARNING
# ============================================================
# One frozen backbone, and the same classifier head as the fusion model, so
# the comparison isolates the effect of fusing two extractors versus one.

transfer_base = MobileNetV2(weights="imagenet", include_top=False,
                            input_shape=(*MODEL_IMAGE_SIZE, 3))
transfer_base.trainable = False

transfer_inputs = tf.keras.Input(shape=(32, 32, 3), name="image")
x = tf.keras.layers.Resizing(*MODEL_IMAGE_SIZE, interpolation="bilinear",
                             name="transfer_resize")(transfer_inputs)
x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1.0,
                              name="transfer_preprocessing")(x)
transfer_features = tf.keras.layers.GlobalAveragePooling2D()(
    transfer_base(x, training=False))

transfer_extractor = tf.keras.Model(transfer_inputs, transfer_features,
                                    name="transfer_extractor")

print("Extracting MobileNetV2 features")
transfer_train = extract_features(transfer_extractor, x_train_fit, "train")
transfer_validation = extract_features(transfer_extractor, x_validation, "validation")

tf.keras.utils.set_random_seed(RANDOM_SEED)
transfer_head = make_classifier_head(transfer_features.shape[-1], "transfer_head")
transfer_head.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=TRANSFER_LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

start_time = time.perf_counter()
history_transfer = transfer_head.fit(
    transfer_train, y_train_fit,
    validation_data=(transfer_validation, y_validation),
    epochs=HEAD_EPOCHS, batch_size=BATCH_SIZE, verbose=0,
    callbacks=[TrainSetEvaluation(transfer_train, y_train_fit)],
).history
training_time_transfer = time.perf_counter() - start_time
print(f"Total training time: {training_time_transfer:.2f} seconds")

model_transfer = tf.keras.Model(transfer_inputs, transfer_head(transfer_features),
                                name="mobilenetv2_transfer")
model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=TRANSFER_LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)
print_model_parameters(model_transfer)

plot_training_history(history_transfer, "Transfer Learning")
transfer_results, y_pred_transfer, cm_transfer = evaluate_classifier(
    model_transfer, "Transfer Learning", x_test, y_test, training_time_transfer)
visualize_predictions("Transfer Learning", x_test, y_test, y_pred_transfer)


# ============================================================
# 6. (c) FINE-TUNING
# ============================================================
# Here the backbone itself is updated, so the features change every epoch and
# cannot be cached: this model trains end to end. Because the backbone learns,
# data augmentation earns its place here (it only regularised the head in the
# two frozen-backbone methods). The classifier starts from the transfer
# learning weights, so fine-tuning continues that model rather than restarting.

finetuning_base = MobileNetV2(weights="imagenet", include_top=False,
                              input_shape=(*MODEL_IMAGE_SIZE, 3))
finetuning_base.trainable = True
for layer in finetuning_base.layers:
    layer.trainable = False
# Open only the last layers, and keep BatchNormalization frozen so its running
# statistics are not corrupted by the small fine-tuning batches.
for layer in finetuning_base.layers[-FINE_TUNE_LAYERS:]:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

print(f"Backbone layers: {len(finetuning_base.layers)}, "
      f"trainable: {sum(layer.trainable for layer in finetuning_base.layers)}")

finetuning_inputs = tf.keras.Input(shape=(32, 32, 3), name="image")
x = make_data_augmentation("finetuning_augmentation")(finetuning_inputs)
x = tf.keras.layers.Resizing(*MODEL_IMAGE_SIZE, interpolation="bilinear")(x)
x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1.0)(x)
x = finetuning_base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(HEAD_DROPOUT, name="finetuning_dropout")(x)
finetuning_outputs = tf.keras.layers.Dense(
    NUMBER_OF_CLASSES, activation="softmax", name="class_output")(x)

model_finetuning = tf.keras.Model(finetuning_inputs, finetuning_outputs,
                                  name="mobilenetv2_finetuned")
# Continue from the transfer learning classifier.
model_finetuning.get_layer("class_output").set_weights(
    transfer_head.layers[-1].get_weights())

# A constant learning rate leaves the validation loss bouncing once the model
# has converged. Cosine decay removes that tail wobble.
steps_per_epoch = int(np.ceil(len(x_train_fit) / FINE_TUNING_BATCH_SIZE))
finetuning_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=FINE_TUNING_LEARNING_RATE,
    decay_steps=FINE_TUNING_EPOCHS * steps_per_epoch,
    alpha=0.02,
)
model_finetuning.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=finetuning_schedule),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)
print_model_parameters(model_finetuning)

start_time = time.perf_counter()
history_finetuning = model_finetuning.fit(
    x_train_fit, y_train_fit,
    validation_data=(x_validation, y_validation),
    epochs=FINE_TUNING_EPOCHS, batch_size=FINE_TUNING_BATCH_SIZE, verbose=1,
    # Measured on a fixed 10k subset: a full pass over 40k images every epoch
    # would roughly double the training time for no extra insight.
    callbacks=[TrainSetEvaluation(x_train_fit[:10000], y_train_fit[:10000], batch_size=128)],
).history
training_time_finetuning = time.perf_counter() - start_time
print(f"Total training time: {training_time_finetuning:.2f} seconds")

plot_training_history(history_finetuning, "Fine-tuning")
finetuning_results, y_pred_finetuning, cm_finetuning = evaluate_classifier(
    model_finetuning, "Fine-tuning", x_test, y_test, training_time_finetuning)
visualize_predictions("Fine-tuning", x_test, y_test, y_pred_finetuning)


# ============================================================
# 7. (d) COMPARISON OF THE METHODS
# ============================================================

comparison = pd.DataFrame(
    [fusion_results, transfer_results, finetuning_results]
).set_index("Model")

display(comparison.round(4))

best_model = comparison["Macro F1"].idxmax()
print(f"Best model according to Macro F1: {best_model} "
      f"({comparison.loc[best_model, 'Macro F1']:.4f})")

bar_colors = [MODEL_COLORS[name] for name in comparison.index]
kpi_columns = ["Accuracy", "Macro precision", "Macro recall", "Macro F1"]


def metric_axis_limits(values):
    """Zoom the axis on the bars, leaving room for the value labels."""
    lowest, highest = float(values.min()), float(values.max())
    margin = max(highest - lowest, 0.01) * 0.5
    return max(0.0, lowest - margin), highest + 2 * margin


def label_bars(axis, format_value):
    """Write the value on top of every bar of an axis."""
    for container in axis.containers:
        axis.bar_label(
            container,
            labels=[format_value(bar.get_height()) for bar in container],
            padding=2, fontsize=8,
        )


# Cost KPIs. Training time is NOT like for like: fusion and transfer learning
# train a small head on cached features, while fine-tuning runs the whole
# backbone every epoch. Inference time is comparable, because every method is
# timed on its full assembled model.
cost_columns = [
    ("Training time (s)", "Training time\n(not like-for-like, see note)",
     "Seconds", lambda v: f"{v:,.0f}"),
    ("Trainable parameters", "Trainable parameters",
     "Number of parameters", lambda v: f"{v:,.0f}"),
    ("Inference (ms/image)", "Inference time",
     "Milliseconds per image", lambda v: f"{v:.3f}"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (column, title, y_label, format_value) in zip(axes, cost_columns):
    ax.bar(comparison.index, comparison[column], color=bar_colors, width=0.6)
    label_bars(ax, format_value)
    ax.set_title(title)
    ax.set_ylabel(y_label)
    ax.set_ylim(0, comparison[column].max() * 1.15)
    ax.tick_params(axis="x", rotation=12)
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)
fig.suptitle("Cost of each method", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.94])
save_figure(fig, "comparison", "cost_kpis")
plt.show()


# ============================================================
# 8. THE THREE FIGURES USED IN THE REPORT
# ============================================================

METHODS = [
    ("Model Fusion", history_fusion, cm_fusion),
    ("Transfer Learning", history_transfer, cm_transfer),
    ("Fine-tuning", history_finetuning, cm_finetuning),
]

# --- Figure 1: learning curves of the three methods, (a) to (c) ---
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# Each row shares one y range across the three methods. With per-panel
# autoscaling the three accuracy panels spanned different intervals, so
# curves of visually similar height meant different numbers and the figure
# could not be read as a comparison.
row_limits = {}
for metric in ("accuracy", "loss"):
    values = np.concatenate([
        np.concatenate([history[f"train_{metric}"], history[f"val_{metric}"]])
        for _, history, _ in METHODS
    ])
    padding = 0.05 * (values.max() - values.min())
    row_limits[metric] = (values.min() - padding, values.max() + padding)

for column, (method_name, method_history, _) in enumerate(METHODS):
    epochs = np.arange(1, len(method_history["val_loss"]) + 1)
    for row, (metric, y_label) in enumerate([("accuracy", "Accuracy"), ("loss", "Loss")]):
        axis = axes[row, column]
        axis.set_ylim(*row_limits[metric])
        axis.plot(epochs, method_history[f"train_{metric}"], color=TRAINING_COLOR,
                  marker="o", markersize=3, linewidth=2, label="Training")
        axis.plot(epochs, method_history[f"val_{metric}"], color=VALIDATION_COLOR,
                  marker="o", markersize=3, linewidth=2, label="Validation")
        axis.set_xlabel("Epoch")
        axis.grid(alpha=0.25)
        axis.set_axisbelow(True)
        axis.legend(fontsize=9)
        if column == 0:
            axis.set_ylabel(y_label)
        else:
            axis.set_yticklabels([])
        if row == 0:
            axis.set_title(method_name, fontsize=13, fontweight="bold")
fig.suptitle("CIFAR-10 - learning curves of the three methods", fontsize=15)
fig.tight_layout(rect=[0, 0, 1, 0.95])
save_figure(fig, "report", "1_learning_curves")
plt.show()

# --- Figure 2: KPI comparison, (d) ---
fig, ax = plt.subplots(figsize=(11, 5.5))
kpi_table = comparison[kpi_columns]
kpi_table.T.plot(kind="bar", ax=ax, rot=0, width=0.75, color=bar_colors)
label_bars(ax, lambda value: f"{value:.3f}")
ax.set_ylim(*metric_axis_limits(kpi_table.values))
ax.set_xlabel("")
ax.set_ylabel("Score")
ax.set_title("CIFAR-10 test-set KPI comparison of the three methods", fontsize=13)
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False)
fig.tight_layout()
save_figure(fig, "report", "2_kpi_comparison")
plt.show()

# --- Figure 3: confusion matrices side by side, (d) ---
fig, axes = plt.subplots(1, 3, figsize=(19, 6))
for axis, (method_name, _, method_cm) in zip(axes, METHODS):
    sn.heatmap(method_cm, xticklabels=class_names, yticklabels=class_names,
               annot=True, fmt=".2f", annot_kws={"size": 7}, cmap="Blues",
               vmin=0, vmax=1, cbar=False, ax=axis)
    axis.set_title(f"{method_name} - accuracy {comparison.loc[method_name, 'Accuracy']:.3f}",
                   fontsize=12, fontweight="bold")
    axis.set_xlabel("Predicted Label")
    axis.set_ylabel("True Label")
fig.suptitle("CIFAR-10 - normalized confusion matrices on the test set", fontsize=15)
fig.tight_layout(rect=[0, 0, 1, 0.94])
save_figure(fig, "report", "3_confusion_matrices")
plt.show()

Python: /home/havard/hand_in/MAS512_Assignment_1/.venv/bin/python
TensorFlow: 2.20.0
Detected GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Physical devices cannot be modified after being initialized
 22462464/170498071 ━━━━━━━━━━━━━━━━━━━━ 29:22 12us/step

KeyboardInterrupt: 

# Q6. Autoencoder                                                           10 marks
Given the data-set fault-classification-class, 
Use Auto encoder to compress the data to latent space and reconstruct it back. Show the following for random 10 iamges from each class
(a) Original image
(b) Latent vector
(c) Reconstructed back image
(d) individual reconstruction error and mean loss for each class.

In [ ]:
# ============================================================
# Q6: AUTOENCODER
# ============================================================

import os

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

# ============================================================
# 1. GENERAL SETTINGS
# ============================================================

n_row = 60
n_col = 175
image_size = (n_row, n_col)
batch_size = 8
encoding_dim = 32
random_seed = 120

# Training settings of the autoencoder. The training set holds 1440 images, so
# a small batch is needed to get enough weight updates per epoch, and a small
# learning rate keeps the ReLU units of the latent layer alive. With the old
# 10 epochs at batch 256 there were only 60 updates in total and most latent
# units stayed dead.
autoencoder_epochs = 100
autoencoder_batch_size = 32
learning_rate = 1e-4
TRAINING_DIRECTORY = "fault-classification-class/training"
TESTING_DIRECTORY = "fault-classification-class/testing"
FIGURE_DIRECTORY = "q6_figures"
FIGURE_DPI = 300

TRAINING_COLOR = "#2A6FBB"
VALIDATION_COLOR = "#E07A1F"
BEST_EPOCH_COLOR = "#C0392B"

np.random.seed(random_seed)
tf.keras.utils.set_random_seed(random_seed)
os.makedirs(FIGURE_DIRECTORY, exist_ok=True)


def save_figure(figure, filename):
    output_path = os.path.join(FIGURE_DIRECTORY, filename)
    figure.savefig(
        output_path,
        dpi=FIGURE_DPI,
        bbox_inches="tight",
        facecolor="white",
    )
    print(f"Figure saved: {output_path}")


def safe_filename(text):
    return "".join(
        character.lower() if character.isalnum() else "_"
        for character in text
    ).strip("_")

# ============================================================
# 2. LOAD THE FAULT CLASSIFICATION DATASET
# ============================================================

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAINING_DIRECTORY,
    labels="inferred",
    color_mode="rgb",
    batch_size=batch_size,
    image_size=image_size,
    shuffle=True,
    seed=random_seed,
    validation_split=0.2,
    subset="training",
)
# shuffle must stay True here, and match the value used for the training
# subset. image_dataset_from_directory shuffles the *file list* before cutting
# it into the two subsets, so the flag decides which files end up where, not
# just the order they come out in. With shuffle=False the folders are read in
# order and the "last 20%" was 360 images that were all of class 10_15, 297 of
# which were also in the shuffled training subset. The validation loss was then
# measured on one class only, on images the autoencoder had already trained on.
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAINING_DIRECTORY,
    labels="inferred",
    color_mode="rgb",
    batch_size=batch_size,
    image_size=image_size,
    shuffle=True,
    seed=random_seed,
    validation_split=0.2,
    subset="validation",
)
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TESTING_DIRECTORY,
    labels="inferred",
    color_mode="rgb",
    batch_size=batch_size,
    image_size=image_size,
    shuffle=False,
)
class_names = train_ds.class_names
print("The classes are:", class_names)
# Use readable names when the folders are 00_15 and 10_15.
class_name_mapping = {
    "00_15": "Healthy",
    "10_15": "Faulty",
}
display_class_names = [
    class_name_mapping.get(class_name, class_name)
    for class_name in class_names
]
print("Displayed class names:", display_class_names)

# ============================================================
# 3. CONVERT THE DATASETS TO NUMPY ARRAYS
# ============================================================


def dataset_to_numpy(dataset):
    """
    Convert a TensorFlow dataset into image and label arrays.
    """
    images_list = []
    labels_list = []
    for images, labels in dataset:
        images_list.append(images.numpy())
        labels_list.append(labels.numpy())
    images = np.concatenate(images_list, axis=0)
    labels = np.concatenate(labels_list, axis=0)
    return images, labels


x_train, y_train = dataset_to_numpy(train_ds)
x_val, y_val = dataset_to_numpy(val_ds)
x_test, y_test = dataset_to_numpy(test_ds)
print("Original train shape:", x_train.shape)
print("Original validation shape:", x_val.shape)
print("Original test shape:", x_test.shape)

# ============================================================
# 4. NORMALIZE AND FLATTEN THE IMAGES
# ============================================================

# Convert pixel values from [0, 255] to [0, 1].
x_train = x_train.astype("float32") / 255.0
x_val = x_val.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
# Total number of pixels in one RGB image.
input_dimension = n_row * n_col * 3
# Flatten each image:
# (60, 175, 3) -> (31500,)
x_train = x_train.reshape((len(x_train), input_dimension))
x_val = x_val.reshape((len(x_val), input_dimension))
x_test = x_test.reshape((len(x_test), input_dimension))
print("Flattened train shape:", x_train.shape)
print("Flattened validation shape:", x_val.shape)
print("Flattened test shape:", x_test.shape)
print("Input dimension:", input_dimension)
print("Latent dimension:", encoding_dim)
print(
    "Compression factor:",
    round(input_dimension / encoding_dim, 2),
)

# ============================================================
# 5. CREATE THE AUTOENCODER
# ============================================================

# This is our input image.
input_img = tf.keras.Input(
    shape=(input_dimension,),
    name="input_image",
)
# This is the encoded representation of the input.
encoded = tf.keras.layers.Dense(
    encoding_dim,
    activation="relu",
    name="latent_vector",
)(input_img)
# This is the lossy reconstruction of the input.
decoded = tf.keras.layers.Dense(
    input_dimension,
    activation="sigmoid",
    name="reconstructed_image",
)(encoded)
# This model maps an input image to its encoded representation.
encoder = tf.keras.Model(
    inputs=input_img,
    outputs=encoded,
    name="encoder",
)
# This model maps an input image to its reconstruction.
autoencoder = tf.keras.Model(
    inputs=input_img,
    outputs=decoded,
    name="autoencoder",
)
autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
    loss="binary_crossentropy",
)
autoencoder.summary()
encoder.summary()

# ============================================================
# 6. TRAIN THE AUTOENCODER
# ============================================================

history = autoencoder.fit(
    x_train,
    x_train,
    epochs=autoencoder_epochs,
    batch_size=autoencoder_batch_size,
    shuffle=True,
    validation_data=(x_val, x_val),
    verbose=2,
)

# ============================================================
# 7. PLOT TRAINING AND VALIDATION LOSS
# ============================================================

loss_epochs = np.arange(1, len(history.history["loss"]) + 1)
training_loss = np.asarray(history.history["loss"])
validation_loss = np.asarray(history.history["val_loss"])
best_epoch = int(np.argmin(validation_loss)) + 1

loss_figure, loss_axis = plt.subplots(figsize=(10, 5.5))

loss_axis.plot(loss_epochs, training_loss, color=TRAINING_COLOR,
               linewidth=2.2, label="Training loss")
loss_axis.plot(loss_epochs, validation_loss, color=VALIDATION_COLOR,
               linewidth=2.2, label="Validation loss")
# The epoch axis is clamped to the data, so the curves run from the left
# spine to the right one instead of floating between two empty margins.
# Ticks are placed by a locator at a constant step of 10 epochs, with a minor
# tick every 5. The previous "1, then every 10" list made the first gap look
# narrower than all the others, which is what made the axis read wrong.
last_epoch = int(loss_epochs[-1])
tick_step = max(1, int(round(last_epoch / 10)))
loss_axis.set_xlim(1, last_epoch)
loss_axis.xaxis.set_major_locator(plt.MultipleLocator(tick_step))
loss_axis.xaxis.set_minor_locator(plt.MultipleLocator(tick_step / 2))
loss_axis.tick_params(axis="x", which="minor", length=3)

# The best epoch is marked with a labelled point rather than a vertical line:
# with the axis clamped to the data, a line at the last epoch would fall on
# the right spine and disappear.
loss_axis.scatter(best_epoch, validation_loss[best_epoch - 1],
                  color=BEST_EPOCH_COLOR, s=70, zorder=5, clip_on=False)
loss_axis.annotate(
    f"Best epoch: {best_epoch}\nvalidation loss {validation_loss[best_epoch - 1]:.4f}",
    xy=(best_epoch, validation_loss[best_epoch - 1]),
    xytext=(-12, 34), textcoords="offset points",
    ha="right", va="bottom", fontsize=9, color=BEST_EPOCH_COLOR,
    arrowprops=dict(arrowstyle="->", color=BEST_EPOCH_COLOR, linewidth=1.2),
)

loss_axis.set_xlabel("Epoch")
loss_axis.set_ylabel("Binary cross-entropy loss")
loss_axis.set_title(
    f"Autoencoder - Training and Validation Loss\n"
    f"final training {training_loss[-1]:.4f}   -   final validation {validation_loss[-1]:.4f}",
    fontsize=13,
)
loss_axis.grid(alpha=0.25)
loss_axis.set_axisbelow(True)
loss_axis.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2, frameon=False)

loss_figure.tight_layout()
save_figure(loss_figure, "01_autoencoder_training_loss.png")
plt.show()


# ============================================================
# 8. CREATE THE DECODER MODEL
# ============================================================

decoder_input = tf.keras.layers.Input(
    shape=(encoding_dim,),
    name="decoder_input",
)
decoder_layer = autoencoder.get_layer("reconstructed_image")
decoder = tf.keras.Model(
    inputs=decoder_input,
    outputs=decoder_layer(decoder_input),
    name="decoder",
)
decoder.summary()

# ============================================================
# 9. ENCODE AND RECONSTRUCT THE TEST IMAGES
# ============================================================

# Latent vectors: shape = (number of test images, encoding_dim)
encoded_imgs = encoder.predict(
    x_test,
    batch_size=256,
    verbose=1,
)
# Reconstructed images: shape = (number of test images, input_dimension)
decoded_imgs = decoder.predict(
    encoded_imgs,
    batch_size=256,
    verbose=1,
)
print("Encoded images shape:", encoded_imgs.shape)
print("Decoded images shape:", decoded_imgs.shape)

# A ReLU unit that is zero for every test image carries no information.
active_units = int(np.sum(encoded_imgs.max(axis=0) > 0))
print(f"Active latent units: {active_units} of {encoding_dim}")

# ============================================================
# 10. INDIVIDUAL RECONSTRUCTION ERRORS
# ============================================================

# Mean Squared Error for every individual test image.
individual_mse = np.mean(
    np.square(x_test - decoded_imgs),
    axis=1,
)
# Binary cross-entropy for every individual test image.
# This is the same type of loss used to train the autoencoder.
epsilon = tf.keras.backend.epsilon()
decoded_imgs_clipped = np.clip(
    decoded_imgs,
    epsilon,
    1.0 - epsilon,
)
individual_bce = -np.mean(
    x_test * np.log(decoded_imgs_clipped)
    + (1.0 - x_test) * np.log(1.0 - decoded_imgs_clipped),
    axis=1,
)
print("Individual MSE shape:", individual_mse.shape)
print("Individual BCE shape:", individual_bce.shape)

# ============================================================
# 11. MEAN RECONSTRUCTION ERROR AND LOSS FOR EACH CLASS
# ============================================================

print("\nMEAN RECONSTRUCTION RESULTS FOR EACH CLASS")
print("=" * 60)
for class_index, class_name in enumerate(display_class_names):
    # Select all test samples belonging to the current class.
    class_mask = y_test == class_index
    class_mean_mse = np.mean(
        individual_mse[class_mask]
    )
    class_mean_bce = np.mean(
        individual_bce[class_mask]
    )
    number_of_samples = np.sum(class_mask)
    print(f"\nClass: {class_name}")
    print(f"Number of test images: {number_of_samples}")
    print(f"Mean reconstruction error (MSE): {class_mean_mse:.8f}")
    print(f"Mean autoencoder loss (BCE): {class_mean_bce:.8f}")

# ============================================================
# 12. SELECT 10 RANDOM IMAGES FROM EACH CLASS
# ============================================================

number_of_images = 10
random_generator = np.random.default_rng(random_seed)
selected_indices_by_class = {}
for class_index, class_name in enumerate(display_class_names):
    class_indices = np.where(y_test == class_index)[0]
    if len(class_indices) < number_of_images:
        raise ValueError(
            f"Class {class_name} contains only "
            f"{len(class_indices)} test images."
        )
    selected_indices = random_generator.choice(
        class_indices,
        size=number_of_images,
        replace=False,
    )
    selected_indices_by_class[class_index] = selected_indices

# ============================================================
# 13. PRINT THE INDIVIDUAL ERRORS OF THE SELECTED IMAGES
# ============================================================

for class_index, class_name in enumerate(display_class_names):
    selected_indices = selected_indices_by_class[class_index]
    print("\n" + "=" * 70)
    print(f"CLASS: {class_name}")
    print("=" * 70)
    for image_number, image_index in enumerate(selected_indices, start=1):
        print(
            f"Image {image_number:2d} | "
            f"Test index: {image_index:4d} | "
            f"MSE: {individual_mse[image_index]:.8f} | "
            f"BCE: {individual_bce[image_index]:.8f}"
        )
    print(
        f"\nMean MSE of the 10 displayed images: "
        f"{np.mean(individual_mse[selected_indices]):.8f}"
    )
    print(
        f"Mean BCE of the 10 displayed images: "
        f"{np.mean(individual_bce[selected_indices]):.8f}"
    )

# ============================================================
# 14. DISPLAY ORIGINAL, LATENT AND RECONSTRUCTED IMAGES
# ============================================================

# One shared scale for every latent bar chart, so the plots are comparable
# between images and between classes.
all_selected_indices = np.concatenate(list(selected_indices_by_class.values()))
latent_maximum = np.max(encoded_imgs[all_selected_indices])

for class_index, class_name in enumerate(display_class_names):
    selected_indices = selected_indices_by_class[class_index]

    results_figure, axes = plt.subplots(
        3,
        number_of_images,
        figsize=(20, 5),
        gridspec_kw={"height_ratios": [0.75, 2.2, 0.75]},
    )

    for i, image_index in enumerate(selected_indices):
        # --- ROW 1: original image ---
        ax = axes[0, i]
        ax.imshow(x_test[image_index].reshape(n_row, n_col, 3))
        ax.set_title(f"Image {i + 1}", fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])

        # --- ROW 2: latent vector, drawn as a bar chart ---
        # Shown as bars rather than a stretched grey strip: the ReLU latent
        # code is sparse, so an image of it is almost entirely black.
        ax = axes[1, i]
        ax.bar(range(encoding_dim), encoded_imgs[image_index], width=0.9, color="#2A6FBB")
        ax.set_ylim(0, latent_maximum * 1.05)
        ax.set_xlim(-0.5, encoding_dim - 0.5)
        ax.set_xticks([])
        ax.tick_params(axis="y", labelsize=8)
        if i > 0:
            ax.set_yticklabels([])
        ax.grid(axis="y", alpha=0.25)

        # --- ROW 3: reconstructed image ---
        ax = axes[2, i]
        ax.imshow(np.clip(decoded_imgs[image_index].reshape(n_row, n_col, 3), 0.0, 1.0))
        ax.set_title(f"MSE = {individual_mse[image_index]:.5f}", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])

    # Horizontal row labels. set_ylabel only works because the ticks are
    # cleared above instead of hiding the whole axis, which also hid the label.
    row_labels = ["Original", f"Latent vector\n({encoding_dim} values)", "Reconstructed"]
    for row, label in enumerate(row_labels):
        axes[row, 0].set_ylabel(
            label,
            rotation=0,
            ha="right",
            va="center",
            labelpad=45 if row == 1 else 10,
            fontsize=11,
            fontweight="bold",
        )

    results_figure.suptitle(f"Autoencoder results - Class: {class_name}", fontsize=16)
    results_figure.tight_layout(rect=[0, 0, 1, 0.94])
    class_file_name = safe_filename(class_name)
    save_figure(
        results_figure,
        f"{class_index + 2:02d}_autoencoder_results_{class_file_name}.png",
    )
    plt.show()


# ============================================================
# 15. RECONSTRUCTION ERROR PER CLASS
# ============================================================

error_figure, axes = plt.subplots(1, 2, figsize=(14, 5))
bar_colors = ["#2A6FBB", "#E07A1F"]
bar_positions = np.arange(number_of_images)
bar_width = 0.4

# The bars first and the class means afterwards, so the legend reads in
# that order too.
for class_index, class_name in enumerate(display_class_names):
    axes[0].bar(
        bar_positions + (class_index - 0.5) * bar_width,
        individual_mse[selected_indices_by_class[class_index]],
        width=bar_width,
        color=bar_colors[class_index],
        label=class_name,
    )

for class_index, class_name in enumerate(display_class_names):
    axes[0].axhline(
        np.mean(individual_mse[y_test == class_index]),
        color=bar_colors[class_index],
        linestyle="--",
        linewidth=1.5,
        label=f"{class_name} - class mean",
    )

axes[0].set_xticks(bar_positions)
axes[0].set_xticklabels([f"{i + 1}" for i in bar_positions])
axes[0].set_xlabel("Displayed image")
axes[0].set_ylabel("Reconstruction error (MSE)")
axes[0].set_title("Individual reconstruction error of the 10 displayed images")
axes[0].set_ylim(0, individual_mse[all_selected_indices].max() * 1.30)
axes[0].grid(axis="y", alpha=0.25)
axes[0].legend(fontsize=9, ncol=2)

# Distribution of the error over the whole test set.
axes[1].hist(
    [individual_mse[y_test == class_index] for class_index in range(len(display_class_names))],
    bins=25,
    color=bar_colors[: len(display_class_names)],
    label=display_class_names,
)
for class_index in range(len(display_class_names)):
    axes[1].axvline(
        np.mean(individual_mse[y_test == class_index]),
        color=bar_colors[class_index],
        linestyle="--",
        linewidth=1.5,
    )

axes[1].set_xlabel("Reconstruction error (MSE)")
axes[1].set_ylabel("Number of test images")
axes[1].set_title("Error distribution over the whole test set")
axes[1].grid(axis="y", alpha=0.25)
axes[1].legend(fontsize=9)

error_figure.tight_layout()
save_figure(error_figure, "04_reconstruction_error_per_class.png")
plt.show()

Found 1800 files belonging to 2 classes.
Using 1440 files for training.
Found 1800 files belonging to 2 classes.
Using 360 files for validation.
Found 200 files belonging to 2 classes.
The classes are: ['00_15', '10_15']
Displayed class names: ['Healthy', 'Faulty']
Original train shape: (1440, 60, 175, 3)
Original validation shape: (360, 60, 175, 3)
Original test shape: (200, 60, 175, 3)
Flattened train shape: (1440, 31500)
Flattened validation shape: (360, 31500)
Flattened test shape: (200, 31500)
Input dimension: 31500
Latent dimension: 32
Compression factor: 984.38


Model: "autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)        │ (None, 31500)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent_vector (Dense)           │ (None, 32)             │     1,008,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reconstructed_image (Dense)     │ (None, 31500)          │     1,039,500 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,047,532 (7.81 MB)

 Trainable params: 2,047,532 (7.81 MB)

...

<Figure size 2000x500 with 30 Axes>

Figure saved: q6_figures/03_autoencoder_results_faulty.png


<Figure size 2000x500 with 30 Axes>

Figure saved: q6_figures/04_reconstruction_error_per_class.png


<Figure size 1400x500 with 2 Axes>